<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer-banner-light.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 03 — Graph Agents

This notebook introduces **Graph Agents** — neurosurfer's DAG-based multi-agent orchestration layer.

Instead of a single agent working in isolation, a **Graph** wires multiple agents, tools, and Python
functions into a directed acyclic graph. Each **node** is one unit of work; **edges** (`depends_on`)
declare data flow. The executor runs nodes in topological order, automatically parallelising
independent branches — and, since the V2 engine, graphs can also **branch, loop, and fan out**.

You'll learn:

1. **Graph primitives** — `Graph`, `GraphNode`, node kinds
2. **Tracing** — one env-var setup; every run lands in Langfuse with token costs
3. **Python API** — build a graph from code
4. **YAML / dict** — the file-based equivalent
5. **React nodes** — tool-using agents as graph nodes
6. **WorkflowPackage / Runner / Registry** — the on-disk production path
7. **Control flow** — the `routes` router (classify-and-branch, 1 of N) + `when:` guards (NEW)
8. **Loops & maps** — iterate-until-good and per-item fan-out (NEW)
9. **GraphBuilder** — fluent programmatic authoring with the same validation as YAML (NEW)

**Model used:** `mistralai/ministral-3-14b-reasoning` served via LM Studio on `localhost:1234`
(any tool-capable local model works — adjust the name below)

---

## Contents
1. [Setup](#1-setup) (+ enabling tracing)
2. [What is a Graph](#2-what-is-a-graph)
3. [Building a Graph with Python](#3-building-a-graph-with-python)
4. [Running with GraphExecutor](#4-running-with-graphexecutor)
5. [Defining Graphs from YAML](#5-defining-graphs-from-yaml)
6. [React Nodes — Agents with Tools](#6-react-nodes--agents-with-tools)
7. [WorkflowPackage — Multi-file Format](#7-workflowpackage--multi-file-format)
8. [WorkflowRunner](#8-workflowrunner)
9. [WorkflowRegistry](#9-workflowregistry)
10. [Control Flow — Guards & Routers](#10-control-flow--guards--routers) **(NEW)**
11. [Loops & Maps](#11-loops--maps) **(NEW)**
12. [GraphBuilder](#12-graphbuilder--programmatic-authoring) **(NEW)**

---

## 1. Setup

Same boilerplate as notebooks 01 and 02.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Point Python at the repo root when running from tutorials/
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

version,1.0.0 | python 3.11.13
os,Linux 7.0.0-28-generic (x86_64)
torch,2.9.1+cu129 CUDA: yes (12.9)
mps,no (built: False)
transformers,4.57.6 sentEmb 5.1.0
accelerate,1.12.0 bnb 0.47.0
gpu,NVIDIA GeForce RTX 3080 Ti


neurosurfer 1.0.0


In [2]:
# ── LM Studio connection ──────────────────────────────────────────
# Make sure LM Studio is running with the model loaded and the local server ON.

LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "mistralai/ministral-3-14b-reasoning"   # adjust if your model name differs
CONTEXT_WINDOW  = 20_000

from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",
    context_window = CONTEXT_WINDOW,
)
print(f"Provider: {provider.model}")
print(f"Tool call style: {provider.capabilities.tool_call_style}")

Provider: mistralai/ministral-3-14b-reasoning
Tool call style: openai


### 1.1 Enable tracing (Langfuse)

Neurosurfer traces every graph run automatically — a **workflow span** with one **node span** per
graph node, and each LLM turn as a **generation** with token usage → cost. No code changes are
needed: tracing activates when Langfuse credentials are present in the environment **before the
first run**.

You need three variables (self-hosted stack on `http://localhost:3000`, or
[Langfuse Cloud](https://cloud.langfuse.com) keys):

```bash
export LANGFUSE_HOST="http://localhost:3000"
export LANGFUSE_PUBLIC_KEY="pk-lf-..."
export LANGFUSE_SECRET_KEY="sk-lf-..."
```

The cell below loads them from the repo's `.env`, then **checks the server is actually running**.
To self-host: `docker compose up -d` with the Langfuse compose stack, open
`http://localhost:3000`, create a project, and copy its API keys.

Later in this notebook we'll also use `traced_run("name")` to group a run's node spans under one
named trace — open **http://localhost:3000 → Traces** after running to explore them.

In [3]:
# ── Enable tracing (Langfuse) ───────────────────────────────────────
# Tracing switches ON automatically when Langfuse credentials are in the
# environment. Set them BEFORE the first run — exporters resolve on first use.

import os, httpx

# Option A — set the three variables explicitly:
# os.environ["LANGFUSE_HOST"]       = "http://localhost:3000"
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."   # Langfuse → Settings → API keys
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."

# Option B (used here) — load them from the repo's .env, keeping keys out of
# the notebook:
env_file = repo_root / ".env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line.startswith("LANGFUSE_") and "=" in line:
            key, _, value = line.partition("=")
            os.environ[key.strip()] = value.strip()

host = os.environ.get("LANGFUSE_HOST", "(unset)")
keys = all(os.environ.get(k) for k in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"))

# Is the Langfuse server actually reachable?
try:
    up = httpx.get(f"{host}/api/public/health", timeout=3.0).status_code == 200
except Exception:
    up = False

TRACING = keys and up
print(f"LANGFUSE_HOST : {host}")
print(f"Keys present  : {keys}")
print(f"Server up     : {up}")
print("→ Tracing " + ("ENABLED — every run below lands in Langfuse" if TRACING
                      else "disabled (runs still work, just untraced)"))

LANGFUSE_HOST : http://localhost:3000
Keys present  : True
Server up     : True
→ Tracing ENABLED — every run below lands in Langfuse


---

## 2. What is a Graph

A **Graph** is a Directed Acyclic Graph (DAG) where each **node** is a discrete unit of work
and **edges** (`depends_on`) define data flow. The executor walks the graph in topological order,
running independent nodes in the same layer in parallel.

```
  graph inputs
       │
       ▼
  [researcher]  ───────────────────────────────┬
       │                                       │
       ▼                                       ▼
  [writer]                            [fact_checker]
       │                                       │
       └───────────► [publisher] ◄─────────────┘
                          │
                          ▼
                       output
```

`researcher` runs first, then `writer` and `fact_checker` run in **parallel** (same layer),
then `publisher` runs once both complete.

### Node kinds

| Kind | Backed by | Typical use |
|---|---|---|
| `base` | Single LLM call (`OneShotAgent`) | Summarise, classify, draft — no tools |
| `react` | Full `AgenticLoop` / `ReactAgent` with tools | Multi-step reasoning, file access, web calls |
| `function` / `python` | Python callable (imported by path) | Deterministic transforms, preprocessing |
| `tool` | Neurosurfer `Tool.run()` directly | Structured data lookup, file write, HTTP |
| `router` **(new)** | Expression or LLM decision | Pick ONE downstream branch; the rest are pruned — §10 |
| `loop` **(new)** | Nested body sub-graph | Repeat until `break_when` or `max_iterations` — §11 |
| `map` **(new)** | Nested body sub-graph | Run once per item of an `over` collection — §11 |
| `subgraph` **(new)** | Nested body sub-graph | Composition: run a body once as one node |
| `input` **(new)** | A human | Pause for a value (pre-supplied, interactive, or resumable) |

Any node can also carry a **`when:` guard** (run only if the expression is truthy), a
**`writes:`** name (publish its output as a stable variable), and an **`on_error:`** fallback
target — all covered in §10.

### Key input field

Every LLM-backed node receives `user_intent` from the graph inputs as the primary task
description, plus the formatted outputs of any `depends_on` nodes as context.

---

## 3. Building a Graph with Python

Import `Graph` and `GraphNode` — both are Pydantic models, so validation runs at construction time.

In [4]:
from neurosurfer.graph import Graph, GraphNode

# ── node 1: research ───────────────────────────────────────────────
researcher = GraphNode(
    id          = "researcher",
    kind        = "base",
    description = "Fact-finding node that condenses core concepts.",
    goal        = "Research the topic and produce exactly 5 key bullet points.",
)

# ── node 2: write ─────────────────────────────────────────────────
writer = GraphNode(
    id          = "writer",
    kind        = "base",
    description = "Writing node that turns bullet points into a polished explanation.",
    goal        = "Write a clear, 2-paragraph explanation from the research notes above.",
    depends_on  = ["researcher"],   # ← receives researcher's output as context
)

# ── graph ─────────────────────────────────────────────────────────
graph = Graph(
    name        = "content_pipeline",
    description = "Research a topic, then write a polished explanation.",
    nodes       = [researcher, writer],
    outputs     = ["writer"],   # ← whose output becomes graph.final["writer"]
)

print(f"Graph : {graph.name}")
print(f"Nodes : {[n.id for n in graph.nodes]}")
print(f"Outputs: {graph.outputs}")
for n in graph.nodes:
    deps = n.depends_on or []
    print(f"  {n.id:15s} kind={n.kind:8s} depends_on={deps}")

Graph : content_pipeline
Nodes : ['researcher', 'writer']
Outputs: ['writer']
  researcher      kind=base     depends_on=[]
  writer          kind=base     depends_on=['researcher']


---

## 4. Running with GraphExecutor

`GraphExecutor` is the low-level DAG runner.  Pass a `Graph` and a `Provider`; call `.run(inputs)`
and get a `GraphExecutionResult` back.

**Inputs** are passed as a dict.  The `user_intent` key carries the primary task description that
every LLM node receives. Any additional keys are listed under *Additional inputs* in each node's
prompt.

In [5]:
from neurosurfer.graph import GraphExecutor

executor = GraphExecutor(graph, provider=provider)

result = executor.run({"user_intent": "Explain how attention mechanisms work in Transformer models"})

print(result.execution_summary())
print(f"Succeeded : {result.succeeded}")
print(f"Errors    : {result.errors or 'none'}")

Graph 'content_pipeline': 2 nodes — 2 ok, 0 failed, 0 skipped
Succeeded : True
Errors    : none


In [6]:
# ── per-node results ────────────────────────────────────────────────
for node_id, nr in result.nodes.items():
    status = "ok" if nr.ok else ("skipped" if nr.skipped else "ERROR")
    print(f"\n── {node_id} [{status}] ({nr.duration_ms} ms) ──")
    if nr.ok:
        print(str(nr.raw_output)[:400])
    else:
        print(f"  {nr.error}")

# ── final outputs ──────────────────────────────────────────────────
print(f"\n{'─' * 60}")
print("Final output (writer node):")
print(result.final.get("writer", "(none)"))


── researcher [ok] (12026 ms) ──
### Attention Mechanisms in Transformer Models

- **Core Idea**:
  - Attention allows the model to focus on relevant parts of input data when processing sequence elements, dynamically adjusting weights based on relationships between positions.

- **Scaled Dot-Product Attention**:
  - Computes attention scores by taking the dot product of "query" and "key" vectors, scaling them (to prevent gradient

── writer [ok] (18033 ms) ──
### How Attention Mechanisms Work in Transformer Models

**Core Principle and Scaled Dot-Product Attention**
Attention mechanisms in Transformer models enable the model to dynamically focus on relevant parts of input data when processing each element in a sequence. The core idea is to compute attention scores by taking the dot product of query (Q) and key (K) vectors, which are scaled (divided by 

────────────────────────────────────────────────────────────
Final output (writer node):
### How Attention Mechanisms Work in Transfo

### GraphExecutionResult fields

| Field / property | Type | What it contains |
|---|---|---|
| `result.nodes` | `dict[node_id, NodeExecutionResult]` | Every node's result |
| `result.final` | `dict[node_id, Any]` | Output of the declared `outputs` nodes |
| `result.errors` | `dict[node_id, str]` | Nodes that errored (empty = all ok) |
| `result.skipped` | `list[str]` | Nodes skipped because an upstream failed |
| `result.succeeded` | `bool` | True iff no errors |
| `result.execution_summary()` | `str` | One-line human summary |

Each `NodeExecutionResult` also carries `duration_ms`, `raw_output`, `error`, and `ok`.

---

## 5. Defining Graphs from YAML

The Python API and YAML format are isomorphic — every `GraphNode` field maps directly to a YAML key.
This matters because the **WorkflowPackage** format stores graphs as `graph.yaml` files on disk.

`load_graph_from_dict(data)` validates a raw dict (just as the Python constructor does).
`load_graph(path)` reads and parses a YAML or JSON file.

In [7]:
# The Python graph from section 3 expressed as a plain dict (mirrors graph.yaml)
graph_dict = {
    "name": "content_pipeline",
    "description": "Research a topic, then write a polished explanation.",
    "nodes": [
        {
            "id": "researcher",
            "kind": "base",
            "goal": "Research the topic and produce exactly 5 key bullet points.",
        },
        {
            "id": "writer",
            "kind": "base",
            "goal": "Write a clear, 2-paragraph explanation from the research notes above.",
            "depends_on": ["researcher"],
        },
    ],
    "outputs": ["writer"],
}

from neurosurfer.graph import load_graph_from_dict

graph_from_dict = load_graph_from_dict(graph_dict)
print(f"Loaded: {graph_from_dict.name} — {len(graph_from_dict.nodes)} nodes")

Loaded: content_pipeline — 2 nodes


In [8]:
# Write it to a YAML file and load it back via load_graph(path)
import yaml
from neurosurfer.graph.engine.loader import load_graph

tmp_yaml = Path("/tmp/tutorial_content_pipeline.yaml")
tmp_yaml.write_text(yaml.dump(graph_dict), encoding="utf-8")
print("Written to:", tmp_yaml)
print("YAML contents:")
print(tmp_yaml.read_text())

graph_from_file = load_graph(tmp_yaml)
print(f"Loaded from file: {graph_from_file.name} — {len(graph_from_file.nodes)} nodes")

Written to: /tmp/tutorial_content_pipeline.yaml
YAML contents:
description: Research a topic, then write a polished explanation.
name: content_pipeline
nodes:
- goal: Research the topic and produce exactly 5 key bullet points.
  id: researcher
  kind: base
- depends_on:
  - researcher
  goal: Write a clear, 2-paragraph explanation from the research notes above.
  id: writer
  kind: base
outputs:
- writer

Loaded from file: content_pipeline — 2 nodes


---

## 6. React Nodes — Agents with Tools

A `react` node runs a full **`AgenticLoop`** (or `ReactAgent` for models without native tool calling)
with access to a subset of the neurosurfer tool pool. Declare the tool names in the node's `tools`
list — the executor selects only those tools for that node.

When using `GraphExecutor` directly you must supply a `ToolPool` and a `ToolContext`.  The higher-
level `WorkflowRunner` (section 8) builds the pool for you automatically from the declared tool names.

In [9]:
from neurosurfer.graph import Graph, GraphNode, GraphExecutor
from neurosurfer.tools import default_pool, AutoApproveIOHandler
from neurosurfer.tools.base import ToolContext

# Shipped with neurosurfer — auto approval handler.
# Agents default to approval="auto"; this handler is only needed for the raw
# ToolContext in §5.
print("AutoApproveIOHandler ready (same as approval=\"auto\").")


# ── graph with a react node followed by a base node ──────────────────
react_graph = Graph(
    name="file_reporter",
    nodes=[
        GraphNode(
            id          = "scout",
            kind        = "react",
            description = "Explores the project and gathers file information.",
            goal        = (
                "Use list_dir to explore the project directory. "
                "Read the README.md and one other key file. "
                "Summarise what you find in bullet points, then call finish."
            ),
            tools       = ["list_dir", "read_file", "finish"],
        ),
        GraphNode(
            id          = "reporter",
            kind        = "base",
            description = "Turns the scout's findings into a short report.",
            goal        = "Write a 3-5 sentence project summary from the information gathered above.",
            depends_on  = ["scout"],
        ),
    ],
    outputs=["reporter"],
)

# ── supply native_tools + tool_ctx to the executor ──────────────────
tool_pool = default_pool().select(["list_dir", "read_file", "finish"])
tool_ctx  = ToolContext(cwd=repo_root, io=AutoApproveIOHandler())

executor = GraphExecutor(
    react_graph,
    provider     = provider,
    native_tools = tool_pool,
    tool_ctx     = tool_ctx,
)

result = executor.run({"user_intent": "Summarise this Python project by reading its key files."})
print(result.execution_summary())

AutoApproveIOHandler ready (same as approval="auto").


→ list_dir({'path': '.', 'pattern': '**/README.md'})


  ↳ .pytest_cache/README.md README.md


→ read_file({'path': 'README.md'})


  ↳ 1 <div align="center"> 2 3 <picture> 4 <source media="(prefers-color-scheme: dark)" srcset="https://raw.githubuserconte…


...

→ read_file({'path': 'pyproject.toml'})


  ↳ 1 [build-system] 2 requires = ["hatchling>=1.27"] 3 build-backend = "hatchling.build" 4 5 [project] 6 name = "neurosurf…



→ finish({'summary': 'Summarized the Neurosurfer project by analyzing its README.md and …)


  ↳ Summarized the Neurosurfer project by analyzing its README.md and pyproject.toml files. The project is a production-rea…


Graph 'file_reporter': 2 nodes — 2 ok, 0 failed, 0 skipped


In [10]:
print("Scout (react node) output:")
print(result.nodes["scout"].raw_output)

print(f"\n{'─' * 60}")
print("Reporter (base node) output:")
print(result.final.get("reporter", "(none)"))

Scout (react node) output:
I found a key file for summarization: `README.md`. Let me also explore another likely central file, such as `setup.py`, `pyproject.toml`, or a main module like `__init__.py` in the neurosurfer package. I will look for these next.## Summary of the Neurosurfer Project

### **README.md**
- **Project Overview**:
  Neurosurfer is a Python framework for building intelligent applications that integrate LLM reasoning, tools, and retrieval. It offers a ready-to-run OpenAI-compatible FastAPI gateway, allowing developers to start with lean functionality and scale as needed.
  
- **Key Features**:
  - AgenticLoop (multi-step tool use), ReactAgent (text-parsing ReAct for models without native tool APIs), and Agent (one-shot).
  - Supports LLM providers like Anthropic Claude, OpenAI, and OpenAI-compatible servers (e.g., Ollama, LM Studio, vLLM).
  - Rich tool ecosystem: web search, sandboxed Python execution, file operations, HTTP, headless browser, and memory tools.
  - R

---

## 7. WorkflowPackage — Multi-file Format

For production use, graphs live on disk as a **WorkflowPackage** directory:

```
my_workflow/
  workflow.yaml          ← manifest: name, version, description, entrypoint
  graph.yaml             ← graph spec (nodes, edges, outputs)
  agents/<id>.yaml       ← optional per-node overrides merged at load time
  nodes/<id>.py          ← Python callables for function/python nodes
  schemas.py             ← Pydantic output-schema classes
```

Only `workflow.yaml` and `graph.yaml` are required. `load_package(path)` loads the directory into
an in-memory `WorkflowPackage`.

In [11]:
import yaml
from pathlib import Path

# ── create the package directory on disk ────────────────────────────
pkg_dir = Path("/tmp/content_pipeline")
pkg_dir.mkdir(parents=True, exist_ok=True)

# workflow.yaml — the manifest
(pkg_dir / "workflow.yaml").write_text(
    yaml.dump({
        "name": "content_pipeline",
        "version": "1.0.0",
        "description": "Research a topic and write a polished explanation.",
        "entrypoint": "graph.yaml",
    }),
    encoding="utf-8",
)

# graph.yaml — the graph spec
(pkg_dir / "graph.yaml").write_text(
    yaml.dump({
        "name": "content_pipeline",
        "nodes": [
            {
                "id": "researcher",
                "kind": "base",
                "goal": "Research the topic and produce exactly 5 key bullet points.",
            },
            {
                "id": "writer",
                "kind": "base",
                "goal": "Write a clear, 2-paragraph explanation from the research notes above.",
                "depends_on": ["researcher"],
            },
        ],
        "outputs": ["writer"],
    }),
    encoding="utf-8",
)

print("Package directory:", pkg_dir)
for f in sorted(pkg_dir.rglob("*")):
    print(f"  {f.relative_to(pkg_dir)}")

Package directory: /tmp/content_pipeline
  graph.yaml
  workflow.yaml


In [12]:
from neurosurfer.graph.workflow import load_package

pkg = load_package(pkg_dir)

print(f"Name       : {pkg.name}")
print(f"Version    : {pkg.version}")
print(f"Description: {pkg.description}")
print(f"Graph nodes: {[n.id for n in pkg.graph.nodes]}")
print(f"Outputs    : {pkg.graph.outputs}")

Name       : content_pipeline
Version    : 1.0.0
Description: Research a topic and write a polished explanation.
Graph nodes: ['researcher', 'writer']
Outputs    : ['writer']


---

## 8. WorkflowRunner

`WorkflowRunner` is the **recommended** way to execute a `WorkflowPackage`. It:

- Builds a `ToolPool` automatically from the tool names declared in the graph nodes
- Patches `sys.path` so function-node callables in `nodes/` can be imported
- Validates required graph inputs before running
- Accepts an optional **progress callback** `(node_id, status, duration_ms)` for live UI updates

In [13]:
from neurosurfer.graph.workflow import WorkflowRunner

runner = WorkflowRunner(
    provider,
    cwd=repo_root,   # working directory for tool contexts
)

result = runner.run(
    pkg,
    inputs={"user_intent": "Explain gradient descent in machine learning"},
)

print(result.execution_summary())
print(f"\n{'─' * 60}")
print(result.final.get("writer", "(none)"))

Graph 'content_pipeline': 2 nodes — 2 ok, 0 failed, 0 skipped

────────────────────────────────────────────────────────────
### Gradient Descent in Machine Learning

**Purpose and Process**
Gradient descent is an optimization algorithm used to minimize the loss function of a machine learning model by iteratively adjusting its parameters (e.g., weights). It begins with an initial guess for these parameters, computes the gradient (derivative) of the loss function, and moves the parameters in the opposite direction of this gradient. This process repeats until the loss stabilizes or a maximum number of iterations is reached. The learning rate controls the step size during each update: if it’s too large, the algorithm may overshoot and diverge; if too small, convergence becomes slow.

**Variants and Applications**
Three primary variants exist:
- **Batch Gradient Descent**: Uses the entire dataset for updates, offering stability but at high computational cost.
- **Stochastic Gradient Descent

In [14]:
# ── progress callback ────────────────────────────────────────────────
# Fired once per node after it completes. Useful for live tables / progress bars.

def on_progress(node_id: str, status: str, duration_ms: int) -> None:
    icon = "\u2713" if status == "ok" else "\u2717"
    print(f"  {icon} {node_id:20s} [{status}]  {duration_ms} ms")

print("Running with progress callback\u2026")
result2 = runner.run(
    pkg,
    inputs    = {"user_intent": "Explain backpropagation in neural networks"},
    progress  = on_progress,
)
print(f"\nDone — {result2.execution_summary()}")

Running with progress callback…


  ✓ researcher           [ok]  14510 ms
  ✓ writer               [ok]  9520 ms

Done — Graph 'content_pipeline': 2 nodes — 2 ok, 0 failed, 0 skipped


---

## 9. WorkflowRegistry

`WorkflowRegistry` is a **persistent store** for workflow packages. It saves each package to
`<registry_dir>/<name>/` and lets you list, retrieve, and delete them by name.

The default registry lives at `.neurosurfer/workflows/` in the current directory (configurable
via `NEUROSURFER_HOME`). Pass an explicit `workflows_dir` for tests or isolated environments.

In [15]:
from neurosurfer.graph.workflow import WorkflowRegistry

# Use a temp dir so the demo doesn't pollute the project registry
registry = WorkflowRegistry(workflows_dir=Path("/tmp/ns_tutorial_registry"))

# ── save the package we created earlier ─────────────────────────────
registry.save(pkg)
print("Saved:", registry.list())

Saved: ['content_pipeline']


In [16]:
# ── retrieve and run from the registry ─────────────────────────────
loaded_pkg = registry.get("content_pipeline")

print(f"Retrieved: {loaded_pkg.name} v{loaded_pkg.version}")
print(f"Nodes    : {[n.id for n in loaded_pkg.graph.nodes]}")

# Run it exactly like a freshly-loaded package
result3 = WorkflowRunner(provider, cwd=repo_root).run(
    loaded_pkg,
    inputs={"user_intent": "Explain the difference between RNNs and Transformers"},
)

print(f"\n{result3.execution_summary()}")
print(f"\n{'─' * 60}")
print(result3.final.get("writer", "(none)"))

Retrieved: content_pipeline v1.0.0
Nodes    : ['researcher', 'writer']



Graph 'content_pipeline': 2 nodes — 2 ok, 0 failed, 0 skipped

────────────────────────────────────────────────────────────
### **RNNs vs. Transformers: Key Differences**

#### **Architecture & Information Processing**
Recurrent Neural Networks (RNNs) process sequences step-by-step, maintaining a hidden state to retain information from previous inputs. However, they often struggle with long sequences due to vanishing gradients, which makes it difficult for them to capture dependencies across significant distances in the data. In contrast, Transformers leverage self-attention mechanisms, allowing them to analyze the entire input sequence simultaneously. This parallel processing eliminates the need for sequential computation, enabling better handling of long-range dependencies and more efficient training.

#### **Performance & Scalability**
RNNs are less efficient for longer sequences because they rely on one-step-at-a-time processing, which slows down training and inference. They also 

In [17]:
# ── registry management ─────────────────────────────────────────────
print("Available workflows:", registry.list())
print("Exists 'content_pipeline':", registry.exists("content_pipeline"))
print("Exists 'missing'          :", registry.exists("missing"))

# Delete it and confirm it's gone
registry.delete("content_pipeline")
print("After delete:", registry.list())

Available workflows: ['content_pipeline']
Exists 'content_pipeline': True
Exists 'missing'          : False
After delete: []


---

## 10. Control Flow — Guards & Routers

Everything so far ran **every node once, in order**. The V2 engine adds real control flow.
New scenario: a **support-desk triage** pipeline — one router decides which of **N** paths a
ticket takes.

### The `routes` router — the router IS the classifier

The simple form needs **no separate classify node and no expressions**: give the router an
instruction (`goal`) and a `routes` dict mapping **labels → target nodes**. The router makes one
LLM call, picks a label, runs that branch, and **prunes every other target**. It's N-way by
construction — three, five, ten branches are still ONE router, never a chain of routers.

- `repair: true` (default) — if the model answers something that matches no label, it's retried
  once with corrective feedback before falling back to `default`.
- `default:` — the branch taken when classification can't be mapped at all.

```yaml
nodes:
  - id: route
    kind: router
    goal: "Route this support ticket by its content: {ticket}"
    routes:               # label → target node (N-way by construction)
      urgent:  escalate
      billing: finance
      routine: reply
    repair: true
    default: reply
  - id: escalate          # every target depends_on the router
    kind: base
    depends_on: [route]
    goal: "Draft a 2-sentence escalation notice for: {ticket}"
  # … finance, reply likewise
outputs: [escalate, finance, reply]
```

### Deterministic variants (no LLM call)

- **`when:` guards** — any node can carry a predicate; falsy → the node is **pruned** (a normal
  not-taken branch, distinct from an error-skip). Joins are **OR**: a downstream node runs if
  *any* incoming branch is live.
- **`cases` router** — `cases: [{when: "<expr>", to: target}]` evaluates sandboxed expressions
  over live state (`inputs.*`, `nodes.<id>`, `vars.*`) in order; first truthy wins, else
  `default`. Free and reproducible — ideal when routing on a `function` node's structured
  output. When an expression must test raw LLM text, use `contains(lower(nodes.x), 'label')`,
  never exact `==` (real model output arrives with whitespace/casing noise).

In [18]:
from neurosurfer.graph import Graph, GraphNode, GraphExecutor
from neurosurfer.observability.run import traced_run

# ── the triage graph: ONE router, N branches ────────────────────────
route = GraphNode(
    id     = "route",
    kind   = "router",
    goal   = "Route this support ticket by its content: {ticket}",
    routes = {                       # label → target node (N-way by construction)
        "urgent":  "escalate",
        "billing": "finance",
        "routine": "reply",
    },
    repair  = True,                  # invalid answer → one corrective retry, then default
    default = "reply",
)
escalate = GraphNode(
    id="escalate", kind="base", depends_on=["route"],
    goal="Draft a 2-sentence escalation notice to the on-call engineer for: {ticket}",
)
finance = GraphNode(
    id="finance", kind="base", depends_on=["route"],
    goal="Draft a short internal note to the billing team about: {ticket}",
)
reply = GraphNode(
    id="reply", kind="base", depends_on=["route"],
    goal="Draft a short, polite customer reply for this routine ticket: {ticket}",
)

triage = Graph(
    name    = "ticket_triage",
    inputs  = [{"name": "ticket", "type": "string", "required": True}],
    nodes   = [route, escalate, finance, reply],
    outputs = ["escalate", "finance", "reply"],
)
executor = GraphExecutor(triage, provider=provider)

# ── one ticket per branch ───────────────────────────────────────────
BRANCHES = ("escalate", "finance", "reply")
tickets = [
    "URGENT: production database is down and customers cannot pay!",
    "I was charged twice for my subscription last month.",
    "Tiny typo on the About page: 'teh' should be 'the'.",
]
for ticket in tickets:
    # traced_run groups this run's node spans under ONE named Langfuse trace
    with traced_run("tutorial:ticket-triage", metadata={"ticket": ticket[:60]}):
        res = executor.run({"ticket": ticket})

    taken  = next(b for b in BRANCHES if not res.nodes[b].skipped)
    pruned = [b for b in BRANCHES if res.nodes[b].skipped]
    print(f"\nTicket : {ticket}")
    print(f"Router : → {taken}   (pruned: {', '.join(pruned)} — skipped, NOT errored)")
    print(f"Output : {str(res.final.get(taken)).strip()[:180]}")

[07/23/26 11:52:51] INFO     Node route: routing to 'escalate' (label: urgent)


Ticket : URGENT: production database is down and customers cannot pay!
Router : → escalate   (pruned: finance, reply — skipped, NOT errored)
Output : ```
ESCALATION NOTICE

**Subject:** URGENT: Production Database Outage - Customer Payments Affected

**Notice:**
The production database is down, preventing customers from making p


[07/23/26 11:52:55] INFO     Node route: routing to 'finance' (label: billing)


Ticket : I was charged twice for my subscription last month.
Router : → finance   (pruned: escalate, reply — skipped, NOT errored)
Output : Here’s a concise internal note draft for the billing team:

---
**Subject:** Query on Duplicate Subscription Charge – [Your Name]

**Key Details:**
- **Issue:** Charged twice for m


[07/23/26 11:53:02] INFO     Node route: routing to 'reply' (label: routine)


Ticket : Tiny typo on the About page: 'teh' should be 'the'.
Router : → reply   (pruned: escalate, finance — skipped, NOT errored)
Output : **Customer Reply:**

**Subject:** Quick Fix for Typo on Our About Page

Hi there,

Thank you for bringing this to our attention! We’ve corrected the typo (replacing “teh” with “the


---

## 11. Loops & Maps

Two iteration constructs, both running a nested **`body`** — a mini-graph of their own:

| Construct | Fields | Semantics |
|---|---|---|
| `loop` | `body`, `max_iterations` (**required** hard ceiling), `until` *or* `break_when`, `accumulate` | Re-runs the body until the stop condition or the ceiling. `accumulate="name"` collects every iteration's output into a list (also readable as `vars.name`). |
| `map` | `body`, `over` (expression → a list), `as`/`item_var`, `concurrency` | Runs the body once per item; each run sees `{item}` and `index`. The node's output is the **ordered list** of per-item results — an implicit gather. |

### Two ways to stop a loop

- **`until`** *(simple, recommended)* — state the stop condition in **plain English**:
  `until: "the review approves the slogan"`. After each iteration an internal judge (one small
  LLM call) reads the body's outputs and answers STOP or CONTINUE. On CONTINUE it also says
  *why* — and that reason reaches the next iteration as **`{feedback}`**, turning the loop into
  *directed refinement* instead of blind retry. Body nodes need no output-format contract at all.
- **`break_when`** *(deterministic)* — a sandboxed expression, evaluated for free: budgets
  (`len(...) < 500`), cursors, `index >= 2`. No LLM call per iteration.

They're mutually exclusive; with neither, the loop runs to `max_iterations`.

**Loop demo** — draft → review → redraft, where the judge's critique visibly steers the next
draft via `{feedback}`.

**Map demo** — fan the same body out over a list of cities, two at a time.

In [19]:
# ── loop: draft → review, stop condition in plain English ──────────
draft = GraphNode(
    id   = "draft",
    kind = "base",
    goal = ("Write a punchy one-line slogan for the product: {product}. "
            "Judge feedback from the previous attempt (if any): {feedback}"),
)
review = GraphNode(
    id         = "review",
    kind       = "base",
    depends_on = ["draft"],
    goal       = ("Critically review the slogan above for {product}. "
                  "Point out anything weak — be a tough reviewer."),
)
refine = GraphNode(
    id             = "refine",
    kind           = "loop",
    max_iterations = 3,                            # hard ceiling — ALWAYS required
    until          = "the review clearly approves of the slogan",   # ← plain English
    accumulate     = "attempts",
    body           = [draft, review],
    body_outputs   = ["draft", "review"],
)

loop_graph = Graph(
    name    = "slogan_refinery",
    inputs  = [{"name": "product", "type": "string", "required": True}],
    nodes   = [refine],
    outputs = ["refine"],
)

with traced_run("tutorial:slogan-refinery"):
    res = GraphExecutor(loop_graph, provider=provider).run(
        {"product": "solar-powered backpack"})

info = res.nodes["refine"].structured_output
print(f"Iterations: {info['iterations']}   broke early: {info['broke_early']}\n")
for attempt, verdict in zip(res.nodes["refine"].raw_output, info.get("judge", [])):
    slogan = str(attempt["draft"]).strip().splitlines()[0][:100]
    print(f"  draft : {slogan}")
    print(f"  judge : {'STOP' if verdict['stop'] else 'CONTINUE'}"
          + (f" — {verdict['reason'][:110]}" if verdict['reason'] else ""))
    print()

[07/23/26 11:53:32] INFO     Node refine: iteration 1 → continue (lacks unique selling point and avoids overused   
                             phrasing)

[07/23/26 11:54:26] INFO     Node refine: iteration 2 → continue (lacking eco focus and USP clarity)

[07/23/26 11:55:19] INFO     Node refine: iteration 3 → continue (Lacks adventure specificity and durability       
                             mention)

Iterations: 3   broke early: False

  draft : ### Judging Feedback and Writing Slogan
  judge : CONTINUE — lacks unique selling point and avoids overused phrasing

  draft : ### Refined Slogan Drafts
  judge : CONTINUE — lacking eco focus and USP clarity

  draft : ### Improved Slogan for Solar-Powered Backpack
  judge : CONTINUE — Lacks adventure specificity and durability mention



In [20]:
# ── map: fan out over a list, gather ordered results ───────────────
tip = GraphNode(
    id   = "tip",
    kind = "base",
    goal = "Write ONE practical travel tip for {item}. One sentence only.",
)
per_city = GraphNode(
    id          = "tips",
    kind        = "map",
    over        = "inputs.cities",   # expression yielding the collection
    item_var    = "item",            # each body run sees the current {item} (YAML: `as: item`)
    concurrency = 2,                 # run up to 2 body copies in parallel
    body        = [tip],
)

map_graph = Graph(
    name    = "city_tips",
    inputs  = [{"name": "cities", "type": "array", "required": True}],
    nodes   = [per_city],
    outputs = ["tips"],
)

cities = ["Tokyo", "Lisbon", "Oslo"]
with traced_run("tutorial:city-tips"):
    res = GraphExecutor(map_graph, provider=provider).run({"cities": cities})

# The map node's output is the ordered list of per-item results (implicit gather)
for city, t in zip(cities, res.nodes["tips"].raw_output):
    print(f"  {city:7s} → {str(t).strip()[:150]}")

  Tokyo   → **Tip:** Always validate your **Suica or Pasmo IC card** at the turnstiles in Tokyo stations—failing to do so can result in fines, even if you intend 
  Lisbon  → **Tip for Lisbon:** Use the **Viva Viagem** card for seamless travel across trams, buses, and trains—load it with credit at any metro station or kiosk
  Oslo    → **Travel tip for Oslo:**
Always pack layers and waterproof clothing, as the weather can change rapidly with rain or sun in a single day.


---

## 12. GraphBuilder — Programmatic Authoring

`GraphBuilder` is a fluent Python API that produces the **same validated IR** as YAML —
`.build()` runs the exact validation gate the YAML loader uses (router targets must exist and
depend on the router, expressions must parse, the DAG must be acyclic). Graphs round-trip
through JSON unchanged, which is the contract the HTTP execution API and the visual UI rely on.

Below: the §10 triage workflow rebuilt in five chained calls.

In [21]:
from neurosurfer.graph import GraphBuilder, load_graph_from_dict

g = (
    GraphBuilder("ticket_triage_v2", description="Route support tickets by content")
    .router("route",
            goal="Route this support ticket by its content: {ticket}",
            routes={"urgent": "escalate", "routine": "reply"},
            default="reply")
    .base("escalate",
          goal="Draft a 2-sentence escalation notice to the on-call engineer for: {ticket}",
          depends_on=["route"])
    .base("reply",
          goal="Draft a short, polite customer reply for this routine ticket: {ticket}",
          depends_on=["route"])
    .input("ticket", type="string", required=True)
    .outputs("escalate", "reply")
    .build()   # ← same validation gate as the YAML loader (targets, routes/cases, DAG)
)
print(f"Built    : {g.name} — {[n.id for n in g.nodes]}")

# JSON round-trip — exactly what the HTTP API (and the upcoming visual UI) consume
as_json = g.model_dump(mode="json")
rebuilt = load_graph_from_dict(as_json)
print(f"Round-trip OK: {rebuilt.name}, routes → {rebuilt.node_map()['route'].routes}")

# And it runs like any other graph
with traced_run("tutorial:triage-v2"):
    res = GraphExecutor(g, provider=provider).run(
        {"ticket": "Checkout crashes with error 500 for every customer right now!"})
print("Branch taken :", "escalate" if not res.nodes["escalate"].skipped else "reply")
print("Output       :", str(res.final.get("escalate") or res.final.get("reply"))[:180])
print("\n→ Open http://localhost:3000 → Traces to explore every `tutorial:*` run.")

Built    : ticket_triage_v2 — ['route', 'escalate', 'reply']
Round-trip OK: ticket_triage_v2, routes → {'urgent': 'escalate', 'routine': 'reply'}


[07/23/26 11:55:24] INFO     Node route: routing to 'escalate' (label: urgent)

Branch taken : escalate
Output       : # Escalation Notice

**Issue:**
Checkout is crashing with HTTP 500 errors for all customers.

**Action Requested:**
Please investigate and resolve this urgent production issue imme

→ Open http://localhost:3000 → Traces to explore every `tutorial:*` run.


---

## Summary

| Concept | API | Typical use |
|---|---|---|
| Define a graph in code | `Graph(name=..., nodes=[GraphNode(...), ...])` | Prototyping, testing |
| Define a graph in YAML | `graph.yaml` loaded by `load_graph(path)` | Production, version-controlled |
| Run a graph directly | `GraphExecutor(graph, provider=...).run(inputs)` | Low-level, full control |
| React node with tools | `GraphNode(kind="react", tools=[...])` + `ToolPool` | Multi-step reasoning in a graph |
| Multi-file package | `workflow.yaml` + `graph.yaml` directory | Shareable, on-disk packages |
| Load a package | `load_package(path)` | Any package directory |
| Run a package | `WorkflowRunner(provider).run(pkg, inputs)` | Recommended production path |
| Persist packages | `WorkflowRegistry(dir).save(pkg)` | Save, list, retrieve by name |
| **Tracing** | set `LANGFUSE_*` env vars (+ `traced_run("name")`) | Every run in Langfuse with token cost |
| **Branching (simple)** | `kind="router"`, `routes={label: target}`, `repair`, `default` | The router classifies and picks 1 of N; others pruned |
| **Branching (deterministic)** | `kind="router"`, `cases=[{when, to}]`, `default` | Expression predicates, no LLM call |
| **Conditional step** | `GraphNode(when="contains(lower(nodes.x), 'yes')")` | Run a node only when a predicate holds |
| **Iterate until good (simple)** | `kind="loop"`, `until="<plain English>"`, `max_iterations` | Judge stops the loop; CONTINUE reasons become `{feedback}` |
| **Iterate until (deterministic)** | `kind="loop"`, `break_when="<expr>"`, `max_iterations` | Budgets, cursors, index checks — no LLM call |
| **Per-item fan-out** | `kind="map"`, `over`, `as`, `concurrency`, `body` | Same body per list item, gathered |
| **Programmatic authoring** | `GraphBuilder(...).router(routes=...).loop(until=...).build()` | Same validation gate as YAML |

**Key ideas**
- `user_intent` is the primary graph input key — every LLM node sees it as its task.
- Nodes in the same topological layer run in **parallel** automatically.
- `depends_on` outputs are injected into the downstream node's context verbatim.
- Prefer the **plain-English forms**: `routes` for branching, `until` for loops — internal
  LLM decisions with `repair`, no expression mini-language. Reach for `cases`/`break_when`
  only when you want deterministic, LLM-free control flow on structured state.
- A pruned branch is **skipped, not errored** — and joins are OR: one live incoming branch
  is enough for a downstream node to run.
- In expressions, guard raw LLM text with `contains(lower(nodes.x), '...')`, never exact equality.
- Loops **always** need `max_iterations`; maps **always** need `over`.
- `WorkflowRunner` builds the `ToolPool` from declared tool names so you don't wire it by hand.
- The registry stores packages as directories — inspect or version-control them directly.

---

## What's Next

| # | Notebook | Topic |
|---|---|---|
| 01 | *Providers, Agents & RAG* | Core building blocks |
| 02 | *Custom Tools* | Define and register your own tools |
| **03** | *Graph Agents* ← **you are here** | Multi-agent DAG workflows + control flow |
| **04** | MCP Servers | Connect external MCP servers; (soon) host workflows |
| 05 | The Gateway Server | Serve agents as OpenAI-compatible endpoints |